# 한국어 Reasoning 튜닝 - Gemma 4 E4B + KOREAson YiSang (성능 최적화판 v3)

## ⚠️ v2에서 v3로의 핵심 수정

**Gemma 4는 2026년 4월 2일 정식 출시된 모델입니다.** v2에서 제가 'gemma-3n으로 바꿔야 한다'고 잘못 판단했던 부분을 전면 수정했습니다.

### Gemma 3 ↔ Gemma 4 차이 (파인튜닝에 직접 영향)

| 항목 | Gemma 3 | Gemma 4 |
|------|---------|---------|
| Role | user / model | **system / user / assistant** (native) |
| Turn 마커 | `<start_of_turn>user` / `<end_of_turn>` | **`<\|turn>user` / `<turn\|>`** |
| Thinking mode | 없음 | **`<\|think\|>` 토큰으로 제어** |
| Thought 채널 | 없음 | `<\|channel>thought\n...<channel\|>` |
| Context | 128K | **E2B/E4B 128K, 26B/31B 256K** |
| Reasoner | 외부 학습 필요 | **기본 내장** (thinking mode configurable) |

### v3 변경 포인트
1. **모델명 복원**: `unsloth/gemma-4-E4B-it`
2. **train_on_responses_only 마커 교체**: `<|turn>user\n` / `<|turn>model\n`
3. **Thinking mode 옵션화**: 기본 off로 SFT, 원하면 on
4. **Multi-turn 대화 시 thought 블록 제거 안내**
5. **⚠️ CUDA 13.2 주의 경고** (GGUF 품질 저하)
6. **2026-04-11 업데이트된 chat template 사용 (최신 transformers/unsloth 필수)**

## 권장 환경
- **A100 40GB** 권장 / L4 가능
- ⚠️ **CUDA 13.2는 피하세요** (Unsloth 공식 경고)

## STEP 1. 패키지 설치 (최신 버전 필수)

Gemma 4는 2026-04-11 업데이트된 chat template을 쓰기 때문에 **최신 `transformers` / `unsloth`가 필수**입니다.
설치 후 **런타임 → 세션 다시 시작** 한 번 눌러주세요.

In [ ]:
!pip install -q --upgrade unsloth
!pip install -q --upgrade transformers trl peft accelerate bitsandbytes datasets wandb

# CUDA 런타임 버전 확인 (13.2면 경고)
import subprocess
try:
    nvcc = subprocess.check_output(["nvcc", "--version"]).decode()
    print(nvcc)
    if "13.2" in nvcc:
        print("⚠️ CUDA 13.2 감지! Gemma 4 GGUF에서 품질 저하 보고됨. 다른 CUDA 버전 권장.")
except Exception:
    pass

## STEP 2. HuggingFace / W&B 로그인

In [ ]:
from huggingface_hub import login
import wandb
from getpass import getpass

HF_TOKEN = getpass("HuggingFace token을 입력하세요: ")
WANDB_KEY = getpass("W&B API key를 입력하세요 (없으면 엔터): ")

login(HF_TOKEN)

if WANDB_KEY.strip():
    wandb.login(key=WANDB_KEY)
    REPORT_TO = "wandb"
else:
    REPORT_TO = "none"
    print("W&B 없이 진행합니다.")

## STEP 3. 설정값

### Gemma 4 reasoning 튜닝에서 주의할 점
- Gemma 4는 **이미 reasoning 모델**입니다. 과도한 학습률/에폭은 오히려 기본 능력을 망칠 수 있어요.
- 그래서 v2보다 LR을 한 번 더 낮췄습니다 (5e-5 → **3e-5**).
- 첫 파일럿은 **thinking mode OFF**로 단순 SFT 권장. 결과 보고 thinking SFT로 넘어가세요.

In [ ]:
from unsloth import FastLanguageModel
import torch
import os
import random
import numpy as np

SEED = 42

# ===== 모델 =====
MODEL_NAME = "unsloth/gemma-4-E4B-it"   # 실제 존재하는 Gemma 4 E4B 모델

# ===== 시퀀스/배치 =====
MAX_SEQ_LENGTH = 4096                    # Gemma 4 E4B는 128K까지 지원. A100 여유 있음.
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM = 4                           # 유효 배치 = 8

# ===== 데이터 =====
TRAIN_SAMPLES = 20000
VAL_RATIO = 0.02
MIN_TOKEN_LEN = 64

# ===== 학습 =====
NUM_EPOCHS = 2
LEARNING_RATE = 3e-5                     # Gemma 4는 이미 reasoner라 보수적으로
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
NEFTUNE_ALPHA = 5

# ===== LoRA =====
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# ===== Thinking mode =====
# 첫 파일럿은 False 권장 (기본 SFT)
# True로 하면 시스템 프롬프트에 <|think|>를 자동 삽입하고, 데이터도 thought/answer 구조여야 함
ENABLE_THINKING_TRAINING = False

# ===== 출력 =====
OUTPUT_DIR = "outputs_gemma4_v3"
RUN_NAME = "ko-reason-gemma4-e4b-v3"

# ===== 재현성 =====
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("=" * 50)
print(f"Model:           {MODEL_NAME}")
print(f"Max seq length:  {MAX_SEQ_LENGTH}")
print(f"Train samples:   {TRAIN_SAMPLES}")
print(f"Effective batch: {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM}")
print(f"LoRA r/alpha:    {LORA_R}/{LORA_ALPHA}")
print(f"Learning rate:   {LEARNING_RATE}")
print(f"Epochs:          {NUM_EPOCHS}")
print(f"NEFTune alpha:   {NEFTUNE_ALPHA}")
print(f"Thinking SFT:    {ENABLE_THINKING_TRAINING}")
print("=" * 50)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## STEP 4. 모델 로드 (4bit 양자화)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"pad_token을 eos_token({tokenizer.eos_token})으로 설정")

print(f"vocab size: {len(tokenizer)}")
print(f"eos_token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"pad_token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")

## STEP 5. LoRA 어댑터 부착

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

## STEP 6. 데이터셋 로드

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("KOREAson/YiSang-HighQuality", split="train")
print(raw_ds)
print("\n--- column names ---")
print(raw_ds.column_names)
print("\n--- sample[0] ---")
print(raw_ds[0])

## STEP 7. 데이터 포맷팅 + 길이 필터링 + train/val 분리

### Gemma 4 관련 주의
- Gemma 4는 **assistant** role을 쓰는데, 데이터가 Gemma 3 스타일(`model`)로 저장돼있어도 `apply_chat_template`이 알아서 처리합니다.
- 다만 thinking mode SFT를 하려면 데이터 자체에 thought/answer가 분리돼있어야 해요.

In [ ]:
def format_chat_example(example):
    """messages 컬럼을 Gemma 4 chat template으로 변환."""
    messages = example["messages"]
    
    # thinking training을 원한다면 system에 <|think|> 추가
    if ENABLE_THINKING_TRAINING:
        has_system = any(m.get("role") == "system" for m in messages)
        if not has_system:
            messages = [{"role": "system", "content": "<|think|>"}] + list(messages)
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

# 1) 섞고 샘플링
ds = raw_ds.shuffle(seed=SEED)
if TRAIN_SAMPLES is not None:
    ds = ds.select(range(min(TRAIN_SAMPLES, len(ds))))

# 2) chat template 적용
ds = ds.map(format_chat_example, remove_columns=raw_ds.column_names)

# 3) 토큰 길이 계산
def add_length(example):
    tokens = tokenizer(example["text"], add_special_tokens=False)["input_ids"]
    return {"n_tokens": len(tokens)}

ds = ds.map(add_length, num_proc=2)

# 4) 길이 분포
lengths = np.array(ds["n_tokens"])
print("=== 토큰 길이 분포 (필터 전) ===")
print(f"  min: {lengths.min()}")
print(f"  p25: {np.percentile(lengths, 25):.0f}")
print(f"  p50: {np.percentile(lengths, 50):.0f}")
print(f"  p75: {np.percentile(lengths, 75):.0f}")
print(f"  p95: {np.percentile(lengths, 95):.0f}")
print(f"  p99: {np.percentile(lengths, 99):.0f}")
print(f"  max: {lengths.max()}")
print(f"  >{MAX_SEQ_LENGTH} 초과: {(lengths > MAX_SEQ_LENGTH).sum()} / {len(lengths)}")
print(f"  <{MIN_TOKEN_LEN} 미만:  {(lengths < MIN_TOKEN_LEN).sum()} / {len(lengths)}")

# 5) 필터링
before = len(ds)
ds = ds.filter(
    lambda ex: MIN_TOKEN_LEN <= ex["n_tokens"] <= MAX_SEQ_LENGTH,
    num_proc=2,
)
after = len(ds)
print(f"\n필터링: {before} → {after} (-{before - after})")

ds = ds.remove_columns(["n_tokens"])

# 6) train/val 분리
split_ds = ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

print(f"\ntrain: {len(train_ds)}")
print(f"eval:  {len(eval_ds)}")
print("\n[train sample preview]")
print(train_ds[0]["text"][:1000])
print("...")

## STEP 8. Gemma 4 chat template 마커 확인

**Gemma 4는 Gemma 3와 완전히 다른 마커를 씁니다.**

- Gemma 3: `<start_of_turn>user\n` / `<start_of_turn>model\n`
- Gemma 4: **`<|turn>user\n`** / **`<|turn>model\n`**

실제 template을 찍어서 마커를 직접 눈으로 확인합니다.

In [ ]:
toy_messages = [
    {"role": "user", "content": "USER_MARKER_12345"},
    {"role": "assistant", "content": "ASSISTANT_MARKER_67890"},
]
toy_text = tokenizer.apply_chat_template(
    toy_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print("=== repr (정확한 문자열) ===")
print(repr(toy_text))
print("\n=== 사람이 읽기 쉽게 ===")
print(toy_text)

# Gemma 4 공식 마커
INSTRUCTION_PART = "<|turn>user\n"
RESPONSE_PART = "<|turn>model\n"

# 검증
if INSTRUCTION_PART in toy_text and RESPONSE_PART in toy_text:
    print("\n✅ Gemma 4 마커 확인 통과")
else:
    print("\n⚠️  마커를 못 찾았습니다. 위 repr 출력을 보고 실제 마커로 교체하세요.")
    print(f"    시도한 INSTRUCTION_PART: {INSTRUCTION_PART!r}")
    print(f"    시도한 RESPONSE_PART:    {RESPONSE_PART!r}")
    print("    unsloth/transformers 버전이 오래됐을 수 있습니다. 최신으로 업그레이드 후 재시도.")

## STEP 9. Trainer 설정

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size = PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio = WARMUP_RATIO,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        bf16 = use_bf16,
        fp16 = not use_bf16,
        max_grad_norm = MAX_GRAD_NORM,
        logging_steps = 10,
        eval_steps = 50,
        save_steps = 50,
        save_strategy = "steps",
        eval_strategy = "steps",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        optim = "adamw_8bit",
        weight_decay = WEIGHT_DECAY,
        lr_scheduler_type = "cosine",
        seed = SEED,
        data_seed = SEED,
        report_to = REPORT_TO,
        run_name = RUN_NAME,
        packing = False,
        neftune_noise_alpha = NEFTUNE_ALPHA,
        dataloader_num_workers = 2,
        group_by_length = True,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 3,
            early_stopping_threshold = 0.001,
        ),
    ],
)

# === 응답부만 학습 (Gemma 4 마커) ===
try:
    from unsloth.chat_templates import train_on_responses_only

    trainer = train_on_responses_only(
        trainer,
        instruction_part = INSTRUCTION_PART,   # <|turn>user\n
        response_part = RESPONSE_PART,         # <|turn>model\n
    )
    print("✅ assistant 응답부만 학습 (Gemma 4 template)")
except Exception as e:
    print("⚠️  train_on_responses_only 적용 실패:", e)
    print("    → 전체 텍스트 학습으로 진행됩니다. 성능 손해 가능성.")
    print("    → STEP 8 repr 출력을 보고 마커를 재확인하세요.")

### 응답부 학습 검증 (추천)

In [ ]:
sample = trainer.train_dataset[0]
if "labels" in sample:
    labels = sample["labels"]
    input_ids = sample["input_ids"]
    n_total = len(labels)
    n_masked = sum(1 for l in labels if l == -100)
    n_trained = n_total - n_masked
    print(f"전체 토큰: {n_total}")
    print(f"마스크(−100): {n_masked}  ({100*n_masked/n_total:.1f}%)")
    print(f"학습 대상:    {n_trained}  ({100*n_trained/n_total:.1f}%)")
    print("\n--- 학습 대상 토큰 미리보기 (최초 100개) ---")
    trained_ids = [t for t, l in zip(input_ids, labels) if l != -100][:100]
    print(tokenizer.decode(trained_ids))
    
    if n_trained / n_total < 0.1:
        print("\n⚠️  학습 대상이 10% 미만입니다. 마커가 안 맞을 가능성!")
    elif n_trained / n_total > 0.9:
        print("\n⚠️  거의 모든 토큰이 학습 대상입니다. responses-only가 적용 안 됐을 수 있음.")
else:
    print("⚠️  labels 컬럼이 없습니다 (응답부 학습 적용 실패).")

## STEP 10. 학습 실행

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    mem_before = torch.cuda.memory_allocated() / 1e9
    print(f"학습 시작 전 GPU 메모리: {mem_before:.2f} GB")

train_result = trainer.train()

print("\n=== 학습 완료 ===")
print(train_result)

print("\n=== 최종 Evaluation ===")
eval_result = trainer.evaluate()
print(eval_result)

## STEP 11. LoRA 저장

In [ ]:
model.save_pretrained("lora_model_gemma4_v3")
tokenizer.save_pretrained("lora_model_gemma4_v3")
print("✅ LoRA 어댑터 저장 완료: ./lora_model_gemma4_v3")

## STEP 12. 추론 테스트 (Gemma 4 스타일)

### Gemma 4 추론 시 주의할 점
1. **Thinking mode를 켤지 끌지 선택 가능** (`<|think|>` 시스템 프롬프트)
2. **멀티턴에서는 이전 thought 블록을 히스토리에 넣지 말 것** — 최종 답변만 유지
3. Thinking 시 `<|channel>thought\n...<channel|>` 블록이 먼저 나오고 그 다음 답변

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "피보나치 수열의 10번째 항을 단계별로 구해줘.",
    "철수는 사과 3개를 가지고 있었고, 영희에게 사과 1개를 받은 뒤 절반을 먹었습니다. 철수에게 남은 사과는 몇 개인가요? 단계별로 설명해주세요.",
    "이진 탐색이 선형 탐색보다 빠른 이유를 시간 복잡도 관점에서 설명해줘.",
]

# thinking mode on/off를 선택
USE_THINKING_AT_INFERENCE = False   # True면 시스템에 <|think|> 삽입

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"[테스트 {i}] {prompt}")
    print("="*60)
    
    msgs = []
    if USE_THINKING_AT_INFERENCE:
        msgs.append({"role": "system", "content": "<|think|>"})
    msgs.append({"role": "user", "content": prompt})
    
    inputs = tokenizer.apply_chat_template(
        msgs,
        return_tensors = "pt",
        add_generation_prompt = True,
    ).to("cuda")
    
    out = model.generate(
        input_ids = inputs,
        max_new_tokens = 1024 if USE_THINKING_AT_INFERENCE else 512,
        temperature = 0.3,
        top_p = 0.9,
        do_sample = True,
        repetition_penalty = 1.05,
        pad_token_id = tokenizer.pad_token_id,
    )
    
    generated = out[0][inputs.shape[1]:]
    print(tokenizer.decode(generated, skip_special_tokens=False))   # 특수 토큰도 보기 위해 False

## STEP 13. (선택) HuggingFace 업로드

In [ ]:
# LoRA만 업로드
# model.push_to_hub("본인HF아이디/ko-reason-gemma4-e4b-lora-v3", token=HF_TOKEN)
# tokenizer.push_to_hub("본인HF아이디/ko-reason-gemma4-e4b-lora-v3", token=HF_TOKEN)

# 머지 16bit
# model.push_to_hub_merged(
#     "본인HF아이디/ko-reason-gemma4-e4b-merged-v3",
#     tokenizer,
#     save_method = "merged_16bit",
#     token = HF_TOKEN,
# )

# GGUF (주의: CUDA 13.2 런타임 피하기)
# model.push_to_hub_gguf(
#     "본인HF아이디/ko-reason-gemma4-e4b-gguf-v3",
#     tokenizer,
#     quantization_method = "q4_k_m",
#     token = HF_TOKEN,
# )

## Gemma 4만의 다음 실험 로드맵

### 1단계: 기본 SFT 검증 (이 노트북)
- `ENABLE_THINKING_TRAINING = False`로 학습 → eval_loss 하락 확인

### 2단계: Thinking SFT
- 데이터셋에 `<thinking>...</thinking>` 같은 thought 섹션이 있다면,
- Gemma 4의 `<|channel>thought\n...<channel|>` 형식으로 변환
- `ENABLE_THINKING_TRAINING = True`로 재학습

### 3단계: DoRA / rsLoRA
- `use_dora=True` 또는 `use_rslora=True` 적용

### 4단계: 데이터 확장
- YiSang-HighQuality 전체 데이터
- 다른 한국어 reasoning 데이터 혼합

### 5단계: DPO / ORPO
- SFT 끝낸 체크포인트에 선호 데이터로 후처리

### 6단계: 평가 벤치마크
- KMMLU, KoBEST, Ko-Math 등 정량 평가
- Gemma 4는 reasoning 기본 내장이라 단순 chat benchmark보다 **reasoning benchmark**로 비교해야 개선 효과가 보임

## 참고 레퍼런스
- Unsloth Gemma 4 공식 가이드: https://unsloth.ai/docs/models/gemma-4
- Gemma 4 모델 카드: https://ai.google.dev/gemma/docs/core/model_card_4
- HuggingFace Gemma 4 블로그: https://huggingface.co/blog/gemma4